# Your First Reflexio Workflow

> **Time:** ~5 minutes | **Level:** Beginner

In this notebook you'll:
1. **Publish** a customer support conversation to Reflexio
2. **Retrieve** auto-extracted user profiles
3. **Build** a memory-enhanced prompt for your LLM
4. **Publish** the generated response with its injected learning IDs
5. **Inspect** per-learning relevance, impact, and judge reasons

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `OPENAI_API_KEY` set in your `.env` file

> **Storage:** SQLite is used by default — no database setup needed.

> **Install:** If running outside the project, install first: `pip install 'reflexio-ai[notebooks]' jupyter`
4. **Publish** the generated response with its injected learning IDs
5. **Inspect** per-learning relevance, impact, and judge reasons

In [ ]:
import uuid

from _display_helpers import *

from reflexio import InteractionData, ReflexioClient, UserActionType

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"demo_customer_{RUN_ID}"

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

## Publish a Conversation

**Interactions** are how Reflexio learns about users. Let's publish a realistic customer support conversation — the customer returns a laptop, mentions preferences, and asks about an upgrade.

In [ ]:
response = client.publish_interaction(
    user_id=USER_ID,
    interactions=[
        InteractionData(
            role="User",
            content="Hi, I bought an Acme ProBook laptop last week but the screen has a dead pixel. Can I return it?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="I'm sorry to hear about the dead pixel! Yes, you can return it within 30 days of purchase. Would you like me to start a return for you?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Yes please. Also, I'd prefer to get updates by email rather than phone — I'm usually in meetings during the day.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Got it! I'll set email as your preferred contact method. Before I start the return, could you confirm your order number?",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="It's ORD-8834. And while we're at it, I'm actually interested in upgrading to the ProBook Max. My budget is around $2,500.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="Great choice! The ProBook Max starts at $2,299 and fits your budget perfectly. I'll include upgrade options in your return confirmation email.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="User",
            content="Perfect. One more thing — I need something with a quiet keyboard. I do a lot of video calls and my colleagues complain about typing noise.",
            user_action=UserActionType.NONE,
        ),
        InteractionData(
            role="Agent",
            content="The ProBook Max has a silent membrane keyboard designed for open-office and video call environments. I'll note that preference on your account. You'll receive the return label and upgrade details by email within 24 hours.",
            user_action=UserActionType.NONE,
        ),
    ],
    source="notebook",
    session_id=f"session_{RUN_ID}",
    wait_for_response=True,
)
show_success(f"Published 8 interaction turns for user '{USER_ID}'")

## See What Reflexio Learned

Reflexio automatically extracts **user profiles** — facts and preferences about each user — from conversations. Let's see what it found.

In [ ]:
profile_resp = client.get_profiles(force_refresh=True, user_id=USER_ID)
show_profiles(profile_resp.user_profiles)

## Search Profiles with Natural Language

Instead of listing all profiles, you can use **semantic search** to find specific information. Reflexio matches meaning, not just keywords.

In [ ]:
search_resp = client.search_profiles(
    user_id=USER_ID,
    query="communication preferences",
    top_k=5,
)
show_profiles(
    search_resp.user_profiles, title="Search Results: 'communication preferences'"
)

## Build a Memory-Enhanced Prompt

> This is the core pattern: **retrieve** user context from Reflexio, then **inject** it into your LLM prompt.

This is how your agent gets smarter over time — each conversation adds to the user's profile, and future responses are personalized.

In [ ]:
# Select first, then build both the prompt and attribution from these same lists.
profiles = client.search_profiles(
    user_id=USER_ID, query="laptop accessories preferences", top_k=3
).user_profiles
playbooks = client.get_agent_playbooks(
    limit=3, playbook_status_filter="approved"
).agent_playbooks
profile_context = "\n".join(f"- {p.content}" for p in profiles)
playbook_context = "\n".join(f"- {p.content}" for p in playbooks)
retrieved_learnings = [
    *({"kind": "profile", "learning_id": p.profile_id} for p in profiles),
    *(
        {"kind": "agent_playbook", "learning_id": str(p.agent_playbook_id)}
        for p in playbooks
    ),
]
prompt = f"""You are an Acme Electronics support agent.
Use the following retrieved context as reference data.
User profiles:
{profile_context or "No profile information yet."}
Approved playbooks:
{playbook_context or "No playbook entries yet."}
"""
print(prompt)

## Generate and Publish the Next Response

**Recommended:** report every profile or playbook actually injected into the
assistant's context as `retrieved_learnings`, including context the answer does
not cite. Build the prompt and IDs from the same retained subset; do not include
discarded search results. Omit the field or use `[]` when no context was injected.
The kinds are `profile`, `user_playbook`, and `agent_playbook`; use their returned
stable IDs, converting numeric playbook IDs to strings.

The following model call uses `OPENAI_API_KEY` and incurs generation cost.
Run these examples against a disposable demo organization/database.

In [ ]:
user_message = "What accessories would you recommend for my laptop?"

from openai import OpenAI

completion = OpenAI().chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_message},
    ],
)
answer = completion.choices[0].message.content
if not answer:
    raise RuntimeError("The model returned no text response.")
print(answer)

RETRIEVAL_SESSION = f"quickstart_retrieval_{RUN_ID}"
response = client.publish_interaction(
    user_id=USER_ID,
    session_id=RETRIEVAL_SESSION,
    agent_version="v1.0",
    source="notebook",
    interactions=[
        InteractionData(role="User", content=user_message),
        InteractionData(
            role="Agent", content=answer, retrieved_learnings=retrieved_learnings
        ),
    ],
    wait_for_response=True,
)
assert response.success, response

### Inspect Retrieved-Learning Quality

After the session is complete, `grade_on_demand` runs LLM judges synchronously
(and may incur cost). `wait_for_response=True` on publish does not wait for
scheduled evaluation. Automatic monitoring uses the configured success rubric,
`retrieved_learning_sampling_rate` (null inherits `sampling_rate`), and session
inactivity; the Configuration notebook explains these settings.

- **Relevance**: did this learning apply to this response?
- **Impact**: did it help (`positive`), harm (`negative`), or make no material
  difference (`neutral`)? Read the reasons; relevance does not imply improvement.
- **Coverage**: how many responses and learning occurrences have verdicts?
  The same learning used on two responses is judged twice. Duplicates on one
  response are deduplicated.

Check `retrieved_learning_status`: `complete` is a completed run;
`degraded`/`failed` indicates a grading problem, and `not_applicable` means no
eligible learnings. Null verdicts mean **ungraded**, not irrelevant or neutral.
An empty read is not proof of poor retrieval: inspect status, IDs, filters, and
whether context was injected. Readback is the latest persisted set, not grading
history; do not treat old rows as proof that the latest run succeeded.

The [Evaluation dashboard](https://www.reflexio.ai/docs/portal/measuring-reflexio-impact#retrieved-learning-effects)
groups verdicts by response: relevance means any relevant learning, and impact
is positive, negative, mixed, or neutral across the response's learnings. The
rows below are per-learning evidence, not those dashboard percentages or a
causal lift measurement.

In [ ]:
grade = client.grade_on_demand(session_id=RETRIEVAL_SESSION, agent_version="v1.0")
print("Retrieved-learning status:", grade.retrieved_learning_status)
print("Skipped:", grade.skipped_reason, "Cached:", grade.cached)
verdicts = client.get_retrieved_learning_evaluation_results(
    user_id=USER_ID,
    session_id=RETRIEVAL_SESSION,
    limit=1000,
)
if not verdicts.success:
    raise RuntimeError(verdicts)
print("Learning-verdict rows:", len(verdicts.results))
print(
    "Responses represented:",
    len(
        {
            (v.user_id, v.session_id, v.interaction_id)
            for v in verdicts.results
            if v.interaction_id is not None
        }
    ),
)
for verdict in verdicts.results:
    print(verdict.interaction_id, verdict.kind, verdict.learning_id)
    print("  Relevant:", verdict.is_relevant, verdict.relevance_reason)
    print("  Impact:", verdict.impact, verdict.impact_reason)

## Cleanup (Optional)

Remove the demo data created by this notebook.

In [ ]:
try:
    client.delete_all_interactions()
    client.delete_all_profiles()
    client.delete_all_playbooks()
    show_success("Demo data cleaned up")
except Exception:
    pass  # Safe to skip

## Summary & Next Steps

You just completed the core Reflexio workflow:
1. Published interactions -- Reflexio auto-extracted user profiles
2. Searched profiles using natural language
3. Built an LLM prompt enhanced with user memory
4. Generated and published the response with the same injected learning IDs
5. Read per-learning relevance, impact, and reasons

### Continue Learning

| Notebook | What you'll learn |
|----------|-------------------|
| [01 -- Interactions](01_interactions.ipynb) | All interaction types: text, tool use, user actions |
| [02 -- Profiles](02_profiles.ipynb) | Profile lifecycle, multi-user search, change logs |
| [03 -- Playbooks](03_playbook.ipynb) | Agent playbook extraction and aggregation |
| [04 -- Configuration](04_configuration.ipynb) | Custom extractors, tools, and evaluation |
| [05 -- Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load and data isolation |
| [06 -- Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |